# Stage 3: Store chunks in Chroma and retrieve by question

This notebook saves chunk embeddings in a **local Chroma** vector database, then retrieves the top matching chunks for a question.

Still no chat LLM and no CLI — only persist + retrieve.

## Why a vector database?

In Stage 2 we kept embeddings in a Python list and scored every chunk by hand. That works for a tiny FAQ set.

A **vector database** (here: **Chroma**) stores vectors on disk and finds nearest neighbors quickly. Same idea — cosine / distance search — with less manual bookkeeping.

Flow in this notebook:

```
Documents → Chunks → Ollama embeddings → Chroma (local) → top-k chunks for a question
```

**Prerequisite:** Ollama running with `nomic-embed-text` (same as Stage 2).

## Load and chunk (same helpers as earlier stages)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "knowledge_base").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

KB_DIR = PROJECT_ROOT / "data" / "knowledge_base"
CHROMA_DIR = PROJECT_ROOT / ".chroma"
COLLECTION_NAME = "harbor_knowledge"

assert KB_DIR.is_dir(), f"Missing knowledge base at {KB_DIR}"


def load_text_documents(folder: Path) -> list[dict]:
    documents = []
    for path in sorted(folder.glob("*.txt")):
        documents.append({"source": path.name, "text": path.read_text(encoding="utf-8").strip()})
    return documents


def chunk_by_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in text.split("\n\n") if p.strip()]


def build_chunk_records(documents: list[dict]) -> list[dict]:
    records = []
    for doc in documents:
        for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
            records.append({"source": doc["source"], "chunk_id": i, "text": chunk})
    return records


chunks = build_chunk_records(load_text_documents(KB_DIR))
print(f"Project root: {PROJECT_ROOT}")
print(f"Chunks: {len(chunks)}")
print(f"Chroma path: {CHROMA_DIR}")

## Embed with Ollama (reuse Stage 2 approach)

In [ ]:
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
EMBED_MODEL = "nomic-embed-text"


def check_ollama() -> None:
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        r.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError("Cannot reach Ollama. Start it with: ollama serve") from exc
    names = {m["name"] for m in r.json().get("models", [])}
    if not any(n.startswith(EMBED_MODEL) for n in names):
        raise RuntimeError(f"Missing model. Run: ollama pull {EMBED_MODEL}")
    print("Ollama OK:", ", ".join(sorted(names)))


def embed_text(text: str) -> list[float]:
    response = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": EMBED_MODEL, "prompt": text},
        timeout=120,
    )
    response.raise_for_status()
    vector = response.json()["embedding"]
    if not vector:
        raise RuntimeError("Empty embedding from Ollama")
    return vector


check_ollama()
print("Embedding dims:", len(embed_text("test")))

## Create a local Chroma collection and add chunks

We pass **our** embeddings into Chroma (`embeddings=...`) so Chroma never calls a cloud API.

Re-running this cell **resets** the collection so the notebook stays reproducible.

In [ ]:
import chromadb

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Drop and recreate so re-runs stay clean for learners
existing = {str(c) for c in client.list_collections()}
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # match Stage 2 cosine thinking
)

ids = []
documents = []
metadatas = []
embeddings = []

for chunk in chunks:
    chunk_id = f"{chunk['source']}::{chunk['chunk_id']}"
    ids.append(chunk_id)
    documents.append(chunk["text"])
    metadatas.append({"source": chunk["source"], "chunk_id": chunk["chunk_id"]})
    embeddings.append(embed_text(chunk["text"]))
    print(f"Indexed {chunk_id}")

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
)

print(f"\nCollection {COLLECTION_NAME!r} now has {collection.count()} items")
print(f"On disk under: {CHROMA_DIR}")

## Retrieve top-k chunks for a question

1. Embed the question with Ollama  
2. Ask Chroma for the nearest stored vectors  
3. Read the returned documents + distances

With cosine space, Chroma returns a **distance** where **smaller is closer** (distance ≈ 1 − similarity).

In [ ]:
def retrieve(question: str, top_k: int = 3) -> dict:
    """Query Chroma using an Ollama embedding of the question."""
    q_vec = embed_text(question)
    return collection.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )


def show_retrieval(question: str, top_k: int = 3) -> None:
    result = retrieve(question, top_k=top_k)
    docs = result["documents"][0]
    metas = result["metadatas"][0]
    dists = result["distances"][0]

    print(f"Question: {question}\n")
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
        similarity = 1.0 - float(dist)  # convenient view when using cosine space
        preview = doc.replace("\n", " ")
        if len(preview) > 140:
            preview = preview[:140] + "..."
        print(
            f"{i}. distance={dist:.3f}  approx_similarity={similarity:.3f}  "
            f"{meta['source']}#{meta['chunk_id']}"
        )
        print(f"   {preview}\n")

## Demo: in-scope questions

In [ ]:
show_retrieval("How many days does a refund take to appear on my statement?")
show_retrieval("How long is express delivery after an order ships?")

## Demo: out-of-scope question

Chroma still returns *something* (nearest neighbors always exist). Notice weaker similarities. Stage 4 will teach the LLM to admit when context is not enough.

In [ ]:
show_retrieval("What is the capital of France?")

in_scope = retrieve("How many days does a refund take to appear on my statement?", top_k=1)
out_of_scope = retrieve("What is the capital of France?", top_k=1)

in_sim = 1.0 - float(in_scope["distances"][0][0])
out_sim = 1.0 - float(out_of_scope["distances"][0][0])
in_source = in_scope["metadatas"][0][0]["source"]

print(f"Best approx_similarity (refund): {in_sim:.3f} from {in_source}")
print(f"Best approx_similarity (unrelated): {out_sim:.3f}")

if in_source == "refund-policy.txt" and in_sim > out_sim:
    print("OK: Chroma retrieval prefers the refund policy for the refund question.")
else:
    print("Unexpected ranking — inspect distances and chunk text.")

## Persistence check

Open the same folder again with a new client — the collection should still be there (useful for the future CLI).

In [ ]:
reopened = chromadb.PersistentClient(path=str(CHROMA_DIR))
reloaded = reopened.get_collection(COLLECTION_NAME)
print(f"Reopened collection count: {reloaded.count()}")
assert reloaded.count() == len(chunks)
print("OK: vectors persisted on disk.")

## What you should notice

- Chroma stores documents, metadata, and embeddings locally under `.chroma/`.
- Querying is: embed question → nearest neighbors → read chunk text.
- Retrieval alone does not generate an answer yet.

## Next stage (not built yet)

**Stage 4:** pass retrieved chunks + the question to a local Ollama **chat** model, ground the answer in context, and clearly say when the documents do not contain enough information.